# G1b — Stage 29: Fresh Projection → Valid AP + Purity

**Problem:** `projected_prototypes_ct_l3l4_warmstart.pt` is stale — it was saved at an
early Phase B epoch, but the checkpoint (`proto_seg_ct_l3l4_warmstart.pth`) is the best-val
model from a later epoch. Loading the stale file overwrites the checkpoint's correct prototype
vectors, producing purity≈0.001–0.07 (vs expected ≈0.5–0.8).

**Fix:** Load the checkpoint WITHOUT the stale projection file, run fresh `PrototypeProjection`
on the training set using the checkpoint's own prototype vectors, then recompute purity/AP.

In [1]:
import sys, os
_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(_root)
sys.path.insert(0, _root)
os.environ.setdefault('PYTORCH_MPS_HIGH_WATERMARK_RATIO', '0.0')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import torch
import torch.utils.data as tud
import numpy as np
import pandas as pd
from pathlib import Path

from src.data.mmwhs_dataset import MMWHSSliceDataset, LABEL_NAMES, NUM_CLASSES
from src.models.proto_seg_net import ProtoSegNet
from src.models.prototype_layer import PrototypeProjection
from src.metrics.proto_quality import (
    compute_purity, compute_per_level_ap, compute_compactness,
    compute_level_dominance, compute_effective_quality,
)

DEVICE   = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
DATA_DIR = 'data/pack/processed_data'
CKPT_DIR = Path('checkpoints')
OUT_DIR  = Path('results/v10')
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODALITY     = 'ct'
PROTO_LEVELS = [3, 4]

print(f'Working dir : {os.getcwd()}')
print(f'Device      : {DEVICE}')
print(f'Output dir  : {OUT_DIR}')

Working dir : /Users/amo/programData/cardiac-proto-segmentation
Device      : mps
Output dir  : results/v10


## 1. Load Stage 29 checkpoint — NO projection file

In [2]:
CKPT_PATH = CKPT_DIR / 'proto_seg_ct_l3l4_warmstart.pth'
STALE_PATH = CKPT_DIR / 'projected_prototypes_ct_l3l4_warmstart.pt'

ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
print(f'Checkpoint: epoch={ckpt["epoch"]}  best_val={ckpt["best_val_dice"]:.4f}')

model = ProtoSegNet(
    n_classes=NUM_CLASSES,
    proto_levels=PROTO_LEVELS,
    use_level_attention=ckpt.get('use_level_attention', False),
).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# Confirm: DO NOT load stale projection
print(f'Stale projection exists: {STALE_PATH.exists()} — NOT loading it')

# Verify prototype norms from checkpoint
for l in PROTO_LEVELS:
    p = model.proto_layers[str(l)].prototypes.data
    norms = p.reshape(-1, p.shape[-1]).norm(dim=-1)
    print(f'  L{l} prototype norms: mean={norms.mean():.2f}  min={norms.min():.2f}  max={norms.max():.2f}')

Checkpoint: epoch=10  best_val=0.8215
Stale projection exists: True — NOT loading it
  L3 prototype norms: mean=11.16  min=9.64  max=12.69
  L4 prototype norms: mean=15.83  min=14.40  max=16.66


## 2. Run fresh PrototypeProjection

Finds nearest real training feature vector for each prototype — consistent with the
current checkpoint's encoder state.

In [3]:
FRESH_PROJ_PATH = CKPT_DIR / 'projected_prototypes_ct_l3l4_warmstart_fresh.pt'

train_ds = MMWHSSliceDataset(DATA_DIR, MODALITY, 'train', augment=False, preload=True)
proj_loader = tud.DataLoader(train_ds, batch_size=32, shuffle=False)
# PrototypeProjection expects (image, label) tuples
wrapped = [(b['image'], b['label']) for b in proj_loader]
print(f'Training batches: {len(wrapped)}')

projector = PrototypeProjection(
    encoder=model.encoder,
    proto_layers=model.proto_layers_dict(),
    device='cpu',
)
print('Running projection (CPU)…')
projector.project(wrapped, save_path=str(FRESH_PROJ_PATH))
print(f'Saved fresh projection → {FRESH_PROJ_PATH}')

Training batches: 106
Running projection (CPU)…


Projected prototypes saved → checkpoints/projected_prototypes_ct_l3l4_warmstart_fresh.pt
Saved fresh projection → checkpoints/projected_prototypes_ct_l3l4_warmstart_fresh.pt


## 3. Load fresh projection into model

In [4]:
proj = torch.load(FRESH_PROJ_PATH, map_location='cpu', weights_only=True)
for level, proto_data in proj['proto_state'].items():
    model.proto_layers[str(level)].prototypes.data.copy_(proto_data)
print('Fresh projection loaded.')

# Verify prototype norms after fresh projection
for l in PROTO_LEVELS:
    p = model.proto_layers[str(l)].prototypes.data
    norms = p.reshape(-1, p.shape[-1]).norm(dim=-1)
    print(f'  L{l} prototype norms after projection: mean={norms.mean():.2f}  min={norms.min():.2f}  max={norms.max():.2f}')

model.to(DEVICE)
model.eval()

Fresh projection loaded.
  L3 prototype norms after projection: mean=6.32  min=5.01  max=8.30
  L4 prototype norms after projection: mean=6.76  min=4.52  max=9.50


ProtoSegNet(
  (encoder): HierarchicalEncoder2D(
    (level1): EncoderBlock(
      (down): Sequential(
        (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (res): ResBlock(
        (conv): Sequential(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (shortcut): Identity()
        (relu): ReLU(inplace=True)
      )
    )
    (level2): EncoderBlock(
      (down): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), paddin

## 4. Compute Purity and AP

In [5]:
train_loader = tud.DataLoader(
    MMWHSSliceDataset(DATA_DIR, MODALITY, 'train', augment=False, preload=True),
    batch_size=16, shuffle=False
)
test_loader = tud.DataLoader(
    MMWHSSliceDataset(DATA_DIR, MODALITY, 'test', augment=False, preload=True),
    batch_size=16, shuffle=False
)

print('Computing purity (train set)…')
purity_df = compute_purity(model, train_loader)

print('Computing AP (test set)…')
ap_df = compute_per_level_ap(model, test_loader)

print('Computing compactness (test set)…')
compact_df = compute_compactness(model, test_loader)

print('Computing level dominance (test set)…')
dom_df = compute_level_dominance(model, test_loader)

print('Computing effective quality…')
eq_df = compute_effective_quality(purity_df, ap_df, compact_df, dom_df)

# Save
purity_df.to_csv(OUT_DIR / 'xai_purity_stage29_fresh.csv', index=False)
ap_df.to_csv(OUT_DIR / 'xai_ap_stage29_fresh.csv', index=False)
compact_df.to_csv(OUT_DIR / 'xai_compactness_stage29_fresh.csv', index=False)
eq_df.to_csv(OUT_DIR / 'xai_effective_quality_stage29_fresh.csv', index=False)

print('\n─── Per-level Purity (fresh projection) ───')
print(purity_df.groupby('level')[['purity']].mean().round(3).to_string())
print('\n─── Per-level AP (fresh projection) ───')
print(ap_df.groupby('level')[['ap']].mean().round(3).to_string())
print('\n─── Effective quality ───')
print(eq_df.round(3).to_string(index=False))

Computing purity (train set)…


Computing AP (test set)…


Computing compactness (test set)…


Computing level dominance (test set)…


Computing effective quality…

─── Per-level Purity (fresh projection) ───
       purity
level        
3       0.381
4       0.744

─── Per-level AP (fresh projection) ───
          ap
level       
3      0.040
4      0.067

─── Effective quality ───
 weight_l3  purity_l3  ap_l3  compact_l3  weight_l4  purity_l4  ap_l4  compact_l4  effective_purity  effective_ap  effective_compactness
     0.599      0.381   0.04       0.791      0.401      0.744  0.067       0.791             0.527         0.051                  0.791


## 5. Final Summary — Stage 29 with fresh projection

In [6]:
# Load existing faithfulness/stability (already valid from notebook 37)
faith_df = pd.read_csv(OUT_DIR / 'xai_faithfulness_stage29.csv')
stab_df  = pd.read_csv(OUT_DIR / 'xai_stability_stage29.csv')
faith_v  = float(faith_df['faithfulness'].iloc[0])
stab_v   = float(stab_df['stability'].iloc[0])

eff_ap   = float(eq_df['effective_ap'].iloc[0])
eff_pur  = float(eq_df['effective_purity'].iloc[0])
eff_comp = float(eq_df['effective_compactness'].iloc[0])
val_dice = ckpt['best_val_dice']

rows = [
    ('Val Dice',           val_dice, None, None, '—'),
    ('Effective Purity',   eff_pur,  None, None, '—'),
    ('Effective AP',       eff_ap,   0.15, 0.25, '✅' if eff_ap  >= 0.15 else '❌'),
    ('Effective Compact.', eff_comp, None, None, '—'),
    ('Faithfulness',       faith_v,  0.15, 0.30, '✅' if faith_v >= 0.15 else '❌'),
    ('Stability',          stab_v,   None, 2.00, '✅' if stab_v  <= 2.00 else '❌'),
]
summary = pd.DataFrame(rows, columns=['Metric', 'Value', 'Min gate', 'Target', 'Pass'])
summary.to_csv(OUT_DIR / 'xai_summary_stage29_fresh.csv', index=False)

print('Stage 29 — Final XAI Summary (fresh projection)')
print('=' * 60)
print(summary.to_string(index=False))
print()

# Comparison table
v9_dir = Path('results/v9')
s9b = pd.read_csv(v9_dir / 'xai_summary_9b.csv').set_index('Metric')['Value']
s9a = pd.read_csv(v9_dir / 'xai_summary_9a.csv').set_index('Metric')['Value']
s29 = summary.set_index('Metric')['Value']

metrics_show = ['Val Dice', 'Effective Purity', 'Effective AP', 'Faithfulness', 'Stability']
print('=' * 75)
print('  Two-Barrier Framework — Updated Comparison')
print('=' * 75)
print(f'  {"Metric":<22}  {"Stage 29 (fresh)":>16}  {"9b (no-skip)":>12}  {"9a (no-skip)":>12}')
print(f'  {"":.<22}  {"L3+L4 skip":>16}  {"L3+L4 noskip":>12}  {"L4 noskip":>12}')
print('  ' + '─' * 70)
for m in metrics_show:
    v29 = float(s29.get(m, float('nan')))
    v9b = float(s9b.get(m, float('nan')))
    v9a = float(s9a.get(m, float('nan')))
    delta = v9b - v29
    arrow = '↑' if delta > 0 else '↓'
    print(f'  {m:<22}  {v29:>16.3f}  {v9b:>12.3f}  {v9a:>12.3f}  Δ={delta:>+.3f}{arrow}')
print('=' * 75)

Stage 29 — Final XAI Summary (fresh projection)
            Metric    Value  Min gate  Target Pass
          Val Dice 0.821452       NaN     NaN    —
  Effective Purity 0.526955       NaN     NaN    —
      Effective AP 0.050857      0.15    0.25    ❌
Effective Compact. 0.791018       NaN     NaN    —
      Faithfulness 0.068851      0.15    0.30    ❌
         Stability 3.381648       NaN    2.00    ❌

  Two-Barrier Framework — Updated Comparison
  Metric                  Stage 29 (fresh)  9b (no-skip)  9a (no-skip)
  ......................        L3+L4 skip  L3+L4 noskip     L4 noskip
  ──────────────────────────────────────────────────────────────────────
  Val Dice                           0.821         0.559         0.606  Δ=-0.263↓
  Effective Purity                   0.527         0.686         0.679  Δ=+0.159↑
  Effective AP                       0.051         0.301         0.312  Δ=+0.250↑
  Faithfulness                       0.069         0.035         0.012  Δ=-0.034↓
  Stab